# Laya multilingual × InjongoIntent Wolof — fine-tuning V1

This notebook adapts Laya's official typed-decisions fine-tuning workflow to
the **official InjongoIntent Wolof train/dev splits**. It trains on REAL train
only, selects and calibrates on dev, and never loads test. The official test is
reserved for the separate frozen benchmark notebook.

Protocol: `train.jsonl → optimization`, `dev.jsonl → selection/calibration`,
`test.jsonl → not used here`.

In [1]:
from pathlib import Path
import json, os, sys, subprocess

def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src/downstream").is_dir():
            return candidate
    raise RuntimeError("Open this notebook from inside the repository checkout")

REPO_ROOT = find_repository_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MODEL_NAME = "convaiinnovations/laya-multilingual"
MODEL_REVISION = "e4e9ddf21a7b1903b7acffd8814ad4307bf63a67"
DATA_DIR = REPO_ROOT / "data/external/injongo_intent_wol"
TRAIN_PATH = DATA_DIR / "train.jsonl"
DEV_PATH = DATA_DIR / "dev.jsonl"
TEST_PATH = DATA_DIR / "test.jsonl"  # frozen; intentionally not loaded here
BASE_MODEL_DIR = REPO_ROOT / "artifacts/downstream/laya_multilingual_base"
PREPARED_DIR = REPO_ROOT / "artifacts/downstream/laya_injongo_wolof_v1"
OUTPUT_DIR = REPO_ROOT / "checkpoints/downstream/laya_normalization_v1"
SEED = 2026

INSTALL_DEPENDENCIES = False
RUN_CPU_DRY_RUN = False
RUN_FULL_TRAINING = True  # change to True only when ready for the GPU run
NPROC_PER_NODE = 1         # set 2 on a 2-GPU Kaggle runtime

print("Repository:", REPO_ROOT)
print("Output:", OUTPUT_DIR)

Repository: c:\Users\thioy\Desktop\School\PFE
Output: c:\Users\thioy\Desktop\School\PFE\checkpoints\downstream\laya_normalization_v1


## Optional dependency installation

The repository requirements are installed first, followed by the pinned Laya
API used by this adaptation. Leave the switch disabled when the environment is
already prepared.

In [2]:
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements-downstream.txt")], check=True)
else:
    print("Dependency installation skipped.")

Dependency installation skipped.


## Download and validate the official files

All three files are downloaded from a pinned Masakhane repository revision.
Only train and dev are parsed below. No new split or random partition is made.

In [3]:
from src.downstream.injongo import download_official_splits, load_split, INJONGO_INTENTS

download_official_splits(DATA_DIR)
train_rows = load_split(TRAIN_PATH, "train")
dev_rows = load_split(DEV_PATH, "dev")

assert len(train_rows) == 2240 and len(dev_rows) == 320
assert set(row["intent"] for row in train_rows) == set(INJONGO_INTENTS)
assert set(row["row_id"] for row in train_rows).isdisjoint(row["row_id"] for row in dev_rows)
print({"train": len(train_rows), "dev": len(dev_rows), "intents": len(INJONGO_INTENTS)})
print("Official test exists but remains unopened:", TEST_PATH.is_file())

{'train': 2240, 'dev': 320, 'intents': 40}
Official test exists but remains unopened: True


In [4]:
import pandas as pd

intent_counts = pd.Series([row["intent"] for row in train_rows]).value_counts().sort_index()
display(intent_counts.rename("train_rows").to_frame())

,train_rows
alarm,56
balance,56
bill_balance,56
book_flight,56
book_hotel,56
calendar_update,56
cancel_reservation,56
car_rental,56
confirm_reservation,56
cook_time,56


## Fetch the pinned Laya multilingual checkpoint

This preserves Laya's standalone layout (`encoder/`, `tokenizer/`,
`rl_agent_config.json`, `model.safetensors`) and applies the tokenizer-config
compatibility helper used by the official notebook.

In [5]:
from src.downstream.laya_model import snapshot_laya_model, load_laya_tokenizer_config

model_dir = snapshot_laya_model(MODEL_NAME, revision=MODEL_REVISION, local_dir=BASE_MODEL_DIR)
tokenizer, laya_config = load_laya_tokenizer_config(model_dir)
laya_config["max_len"] = 1024
laya_config["head_max_len"] = 256
print("Model directory:", model_dir)
print({key: laya_config.get(key) for key in ("max_len", "head_max_len", "encoder_model")})

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Model directory: C:\Users\thioy\Desktop\School\PFE\artifacts\downstream\laya_multilingual_base
{'max_len': 1024, 'head_max_len': 256, 'encoder_model': None}


## Inspect the exact typed-choice conversion

The state is the Wolof utterance. A single `choice` question contains the full
40-intent inventory. Empty option descriptions keep the Laya head compact;
the intent labels themselves are still rendered. The conversion fails if even
one option marker is lost or two options collapse under truncation.

In [6]:
from src.downstream.injongo import make_laya_question, make_laya_state, build_laya_item

for row in train_rows[:3]:
    item = build_laya_item(row, tokenizer, laya_config)
    print("TEXT:", row["text"])
    print("INTENT:", row["intent"])
    print("LAYA STATE:", make_laya_state(row["text"]))
    print("LAYA QUESTION TYPE:", make_laya_question()["t"])
    print("OPTIONS:", list(make_laya_question()["crit"]))
    print("GOLD:", row["intent"])
    print("ENCODED:", {"tokens": len(item["ids"]), **item["option_stats"]})
    print("-" * 80)

TEXT: Ndax mën nga def ab alarm ci jàngukaay bi ci Dakaar ci 11:55 ci suba ci weeru weeru weeru koñ ?
INTENT: alarm
LAYA STATE: {'utterance': 'Ndax mën nga def ab alarm ci jàngukaay bi ci Dakaar ci 11:55 ci suba ci weeru weeru weeru koñ ?'}
LAYA QUESTION TYPE: choice
OPTIONS: ['alarm', 'balance', 'bill_balance', 'book_flight', 'book_hotel', 'calendar_update', 'cancel_reservation', 'car_rental', 'confirm_reservation', 'cook_time', 'exchange_rate', 'food_last', 'freeze_account', 'ingredients_list', 'interest_rate', 'international_visa', 'make_call', 'meal_suggestion', 'min_payment', 'pay_bill', 'pin_change', 'play_music', 'plug_type', 'recipe', 'restaurant_reservation', 'restaurant_reviews', 'restaurant_suggestion', 'share_location', 'shopping_list_update', 'spending_history', 'text', 'time', 'timezone', 'transactions', 'transfer', 'translate', 'travel_notification', 'travel_suggestion', 'update_playlist', 'weather']
GOLD: alarm
ENCODED: {'tokens': 205, 'options': 40, 'options_distinct':

## Prepare train/dev tensors

The official test file is not read by this step. The manifest records this
explicitly. All 2,240 train examples are retained; unlike Laya's source
notebook, no extra calibration slice is carved out of train because the
official dev split already has that role.

In [7]:
import torch
from src.downstream.laya_model import prepare_laya_items

PREPARED_DIR.mkdir(parents=True, exist_ok=True)
train_items = prepare_laya_items(train_rows, tokenizer, laya_config)
dev_items = prepare_laya_items(dev_rows, tokenizer, laya_config)
TRAIN_ITEMS_PATH = PREPARED_DIR / "train_items.pt"
DEV_ITEMS_PATH = PREPARED_DIR / "dev_items.pt"
torch.save(train_items, TRAIN_ITEMS_PATH)
torch.save(dev_items, DEV_ITEMS_PATH)

preparation_manifest = {
    "train_rows": len(train_items), "dev_rows": len(dev_items),
    "test_loaded": False, "intent_count": len(INJONGO_INTENTS),
    "seed": SEED, "model": MODEL_NAME, "model_revision": MODEL_REVISION,
}
(PREPARED_DIR / "preparation_manifest.json").write_text(json.dumps(preparation_manifest, indent=2) + "\n")
print(preparation_manifest)

{'train_rows': 2240, 'dev_rows': 320, 'test_loaded': False, 'intent_count': 40, 'seed': 2026, 'model': 'convaiinnovations/laya-multilingual', 'model_revision': 'e4e9ddf21a7b1903b7acffd8814ad4307bf63a67'}


## Mini forward pass (no training)

This loads the complete base checkpoint and checks one CPU forward pass. It can
take roughly a minute on CPU because the encoder has about 322M parameters.
The returned prediction is not meaningful before task fine-tuning.

In [8]:
from src.downstream.laya_model import mini_forward_dry_run

if RUN_CPU_DRY_RUN:
    dry_run = mini_forward_dry_run(model_dir, train_rows[0], device="cpu")
    print(json.dumps(dry_run, ensure_ascii=False, indent=2))
else:
    print("CPU dry-run skipped.")

CPU dry-run skipped.


## GPU fine-tuning — run this yourself

The command reuses the official RLCD + cross-entropy objective, separate
encoder/head learning rates, exploration schedule, mixed precision, gradient
checkpointing, rolling checkpoints and post-training temperature fitting.

Adaptations: REAL Injongo train only; dev evaluation after each epoch; best
checkpoint selected by dev macro-F1; choice temperature fitted on dev; no test
loading. On Windows/local single-GPU, keep `NPROC_PER_NODE=1`. On Kaggle 2×T4,
set it to 2. Training output is streamed directly in the notebook.

In [9]:
training_arguments = [
    "--model-dir", str(model_dir),
    "--train-items", str(TRAIN_ITEMS_PATH),
    "--dev-items", str(DEV_ITEMS_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--epochs", "4", "--micro-batch", "8", "--eval-batch", "16",
    "--gradient-accumulation", "4", "--seed", str(SEED), "--precision", "fp16",
]
if NPROC_PER_NODE == 1:
    # Avoid torch.distributed.run on single-GPU Windows/Jupyter. It adds no
    # training benefit here and may hide the child process's actual stderr.
    train_command = [
        sys.executable, "-u", "-m", "src.downstream.laya_finetune",
        *training_arguments,
    ]
else:
    train_command = [
        sys.executable, "-u", "-m", "torch.distributed.run", "--standalone",
        f"--nproc_per_node={NPROC_PER_NODE}",
        "-m", "src.downstream.laya_finetune", *training_arguments,
    ]
print(" ".join(train_command))
if RUN_FULL_TRAINING:
    process = subprocess.Popen(
        train_command,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for output_line in process.stdout:
        print(output_line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(
            f"Laya training process failed with exit code {return_code}. "
            "The underlying traceback is printed immediately above."
        )
else:
    print("Training is disabled. Set RUN_FULL_TRAINING=True when ready.")

c:\Users\thioy\AppData\Local\Programs\Python\Python311\python.exe -u -m src.downstream.laya_finetune --model-dir C:\Users\thioy\Desktop\School\PFE\artifacts\downstream\laya_multilingual_base --train-items c:\Users\thioy\Desktop\School\PFE\artifacts\downstream\laya_injongo_wolof_v1\train_items.pt --dev-items c:\Users\thioy\Desktop\School\PFE\artifacts\downstream\laya_injongo_wolof_v1\dev_items.pt --output-dir c:\Users\thioy\Desktop\School\PFE\checkpoints\downstream\laya_normalization_v1 --epochs 4 --micro-batch 8 --eval-batch 16 --gradient-accumulation 4 --seed 2026 --precision fp16
c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Laya Injongo training: 2240 REAL train, 320 dev, 1 GPU(s), 4 

## Verify the selected checkpoint

Run after training. The final directory contains the best dev-macro-F1 weights,
dev-fitted choice temperature, tokenizer/config, history and protocol manifest.

In [10]:
manifest_path = OUTPUT_DIR / "training_manifest.json"
if manifest_path.is_file():
    training_manifest = json.loads(manifest_path.read_text())
    assert training_manifest["test_used"] is False
    display(training_manifest)
else:
    print("No trained checkpoint yet. Run the GPU cell above first.")

{'experiment': 'laya_downstream_normalization_v1',
 'training_data': 'official InjongoIntent/wol train only',
 'selection_data': 'official InjongoIntent/wol dev only',
 'test_used': False,
 'selection_metric': 'dev_macro_f1',
 'best_epoch': 4,
 'best_dev_metrics': {'rows': 320,
  'accuracy': 0.7875,
  'macro_f1': 0.7903351256156984,
  'cross_entropy': 1.00891801007092},
 'choice_temperature_fitted_on_dev': 1.4450664520263672,
 'train_rows': 2240,
 'dev_rows': 320,
 'intent_count': 40,
 'seed': 2026,
 'hyperparameters': {'model_dir': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\artifacts\\downstream\\laya_multilingual_base',
  'train_items': 'c:\\Users\\thioy\\Desktop\\School\\PFE\\artifacts\\downstream\\laya_injongo_wolof_v1\\train_items.pt',
  'dev_items': 'c:\\Users\\thioy\\Desktop\\School\\PFE\\artifacts\\downstream\\laya_injongo_wolof_v1\\dev_items.pt',
  'output_dir': 'c:\\Users\\thioy\\Desktop\\School\\PFE\\checkpoints\\downstream\\laya_normalization_v1',
  'epochs': 4,
  'micro_batc